In [1]:
import pandas as pd
import numpy as np

In [6]:
ratings = pd.read_csv("ml-latest-small/ml-latest-small/ratings.csv")
movies = pd.read_csv("ml-latest-small/ml-latest-small/movies.csv")
tags = pd.read_csv("ml-latest-small/ml-latest-small/tags.csv")

print("Ratings shape:", ratings.shape)
print("Movies shape:", movies.shape)
print("Tags shape:", tags.shape)

Ratings shape: (100836, 4)
Movies shape: (9742, 3)
Tags shape: (3683, 4)


In [7]:
user_counts = ratings.groupby('userId')['rating'].count()
print(user_counts.describe())

movie_counts = ratings.groupby('movieId')['rating'].count()
print(movie_counts.describe())

num_users = ratings['userId'].nunique()
num_movies = ratings['movieId'].nunique()

sparsity = 1 - (len(ratings) / (num_users * num_movies))
print("Sparsity:", round(sparsity, 4))

count     610.000000
mean      165.304918
std       269.480584
min        20.000000
25%        35.000000
50%        70.500000
75%       168.000000
max      2698.000000
Name: rating, dtype: float64
count    9724.000000
mean       10.369807
std        22.401005
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       329.000000
Name: rating, dtype: float64
Sparsity: 0.983


In [8]:
movie_counts = ratings.groupby('movieId')['rating'].count()

# keep movies with at least 5 ratings
valid_movies = movie_counts[movie_counts >= 5].index

ratings = ratings[ratings['movieId'].isin(valid_movies)]

print("New ratings shape:", ratings.shape)

New ratings shape: (90274, 4)


In [9]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(ratings, test_size=0.2, random_state=42)

print("Train size:", train.shape)
print("Test size:", test.shape)

Train size: (72219, 4)
Test size: (18055, 4)


In [10]:
user_item_matrix = train.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

print("User-item matrix shape:", user_item_matrix.shape)

User-item matrix shape: (610, 3650)


In [11]:
print(user_item_matrix.head())

movieId  1       2       3       4       5       6       7       8       \
userId                                                                    
1           4.0     NaN     4.0     NaN     NaN     4.0     NaN     NaN   
2           NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
3           NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
4           NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
5           4.0     NaN     NaN     NaN     NaN     NaN     NaN     NaN   

movieId  9       10      ...  176371  177593  177765  179401  179819  180031  \
userId                   ...                                                   
1           NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN     NaN   
2           NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN     NaN   
3           NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN     NaN   
4           NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN     N

In [15]:
# Naive baseline: average rating per movie
movie_mean_ratings = train.groupby('movieId')['rating'].mean()

top_10_naive = movie_mean_ratings.sort_values(ascending=False).head(10)

# Convert to titles
naive_df = top_10_naive.reset_index()
naive_df = pd.merge(naive_df, movies, on='movieId')

print("Naive Baseline:")
print(naive_df[['title', 'rating']])

Naive Baseline:
                                               title    rating
0                     Trial, The (Procès, Le) (1962)  4.875000
1                      Dead Alive (Braindead) (1992)  4.833333
2        Love Me If You Dare (Jeux d'enfants) (2003)  4.750000
3                Guess Who's Coming to Dinner (1967)  4.750000
4                            Band of Brothers (2001)  4.750000
5                              Secrets & Lies (1996)  4.722222
6   Three Billboards Outside Ebbing, Missouri (2017)  4.714286
7  Man Bites Dog (C'est arrivé près de chez vous)...  4.700000
8  Swept Away (Travolti da un insolito destino ne...  4.666667
9                                       Yi Yi (2000)  4.666667


In [16]:
# Improved baseline: average + minimum number of ratings
movie_stats = train.groupby('movieId').agg({
    'rating': ['mean', 'count']
})

movie_stats.columns = ['mean_rating', 'num_ratings']

# Keep only movies with enough ratings
min_ratings = 50
popular_movies = movie_stats[movie_stats['num_ratings'] >= min_ratings]

top_10_filtered = popular_movies.sort_values(by='mean_rating', ascending=False).head(10)

# Convert to titles
filtered_df = top_10_filtered.reset_index()
filtered_df = pd.merge(filtered_df, movies, on='movieId')

print("\nImproved Baseline:")
print(filtered_df[['title', 'mean_rating', 'num_ratings']])


Improved Baseline:
                                               title  mean_rating  num_ratings
0                   Shawshank Redemption, The (1994)     4.434524          252
1                                  Goodfellas (1990)     4.339806          103
2                              Godfather, The (1972)     4.322581          155
3                               Departed, The (2006)     4.316092           87
4  Dr. Strangelove or: How I Learned to Stop Worr...     4.298611           72
5                            Dark Knight, The (2008)     4.276860          121
6                         Usual Suspects, The (1995)     4.275000          160
7                          American History X (1998)     4.255208           96
8                                  Fight Club (1999)     4.254054          185
9                                  Casablanca (1942)     4.244048           84


In [17]:
# naive
naive_df = top_10_naive.reset_index()
naive_df = pd.merge(naive_df, movies, on='movieId')

# filtered
filtered_df = top_10_filtered.reset_index()
filtered_df = pd.merge(filtered_df, movies, on='movieId')

print("Naive:\n", naive_df[['title', 'rating']])
print("\nFiltered:\n", filtered_df[['title', 'mean_rating', 'num_ratings']])

Naive:
                                                title    rating
0                     Trial, The (Procès, Le) (1962)  4.875000
1                      Dead Alive (Braindead) (1992)  4.833333
2        Love Me If You Dare (Jeux d'enfants) (2003)  4.750000
3                Guess Who's Coming to Dinner (1967)  4.750000
4                            Band of Brothers (2001)  4.750000
5                              Secrets & Lies (1996)  4.722222
6   Three Billboards Outside Ebbing, Missouri (2017)  4.714286
7  Man Bites Dog (C'est arrivé près de chez vous)...  4.700000
8  Swept Away (Travolti da un insolito destino ne...  4.666667
9                                       Yi Yi (2000)  4.666667

Filtered:
                                                title  mean_rating  num_ratings
0                   Shawshank Redemption, The (1994)     4.434524          252
1                                  Goodfellas (1990)     4.339806          103
2                              Godfather, The (197

In [18]:
from sklearn.metrics.pairwise import cosine_similarity

# Fill NaN with 0 (ONLY for similarity computation)
user_item_filled = user_item_matrix.fillna(0)

# Compute similarity between users
user_similarity = cosine_similarity(user_item_filled)

# Convert to DataFrame for easier use
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

print("User similarity shape:", user_similarity_df.shape)

User similarity shape: (610, 610)


In [19]:
def get_similar_users(user_id, n=5):
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)
    return similar_users.iloc[1:n+1]  # skip itself

print(get_similar_users(1))

userId
45     0.307511
91     0.304712
330    0.296323
217    0.295692
266    0.295058
Name: 1, dtype: float64


In [25]:
def recommend_movies_unweighted(user_id, n=10):
    similar_users = get_similar_users(user_id).index
    
    # movies already watched
    user_movies = user_item_matrix.loc[user_id].dropna().index
    
    # movies rated by similar users
    similar_users_movies = user_item_matrix.loc[similar_users]
    
    # average ratings from similar users
    movie_scores = similar_users_movies.mean().dropna()
    
    # remove already watched
    movie_scores = movie_scores[~movie_scores.index.isin(user_movies)]
    
    # top N
    top_movies = movie_scores.sort_values(ascending=False).head(n)
    
    return top_movies

In [36]:
def recommend_movies_weighted(user_id, n=10, k=10):
    # Step 1: get top-k similar users
    sims = user_similarity_df[user_id].sort_values(ascending=False).iloc[1:k+1]
    
    sim_users = sims.index
    sim_scores = sims.values
    
    # Step 2: movies already watched
    user_movies = user_item_matrix.loc[user_id].dropna().index
    
    # Step 3: get similar users' ratings
    sim_matrix = user_item_matrix.loc[sim_users]
    
    # Step 4: mean-centering (IMPORTANT)
    sim_means = sim_matrix.mean(axis=1)
    sim_matrix_centered = sim_matrix.sub(sim_means, axis=0)
    
    # Step 5: weighted sum
    weighted = sim_matrix_centered.mul(sim_scores, axis=0)
    score_sum = weighted.sum(axis=0)
    
    # Step 6: normalization
    sim_sum = (sim_matrix.notna().mul(sim_scores, axis=0)).sum(axis=0)
    scores = (score_sum / sim_sum).dropna()
    
    # Step 7: add back user's mean
    user_mean = user_item_matrix.loc[user_id].mean()
    scores = scores + user_mean
    
    # ✅ Step 8: FIX → clip to valid rating range
    scores = scores.clip(0, 5)
    
    # Step 9: remove already watched movies
    scores = scores[~scores.index.isin(user_movies)]
    
    # Step 10: return top N
    return scores.sort_values(ascending=False).head(n)

In [37]:
# Unweighted
rec_unweighted = recommend_movies_unweighted(1)

df_un = rec_unweighted.reset_index()
df_un.columns = ['movieId', 'score']
df_un = pd.merge(df_un, movies, on='movieId')


# Weighted
rec_weighted = recommend_movies_weighted(1)

df_w = rec_weighted.reset_index()
df_w.columns = ['movieId', 'score']
df_w = pd.merge(df_w, movies, on='movieId')

In [38]:
print("UNWEIGHTED:\n", df_un[['title', 'score']])
print("\nWEIGHTED:\n", df_w[['title', 'score']])

UNWEIGHTED:
                                    title  score
0                          Casino (1995)    5.0
1     Scott Pilgrim vs. the World (2010)    5.0
2                      Zombieland (2009)    5.0
3            Inglourious Basterds (2009)    5.0
4           Bourne Supremacy, The (2004)    5.0
5              Mask of Zorro, The (1998)    5.0
6  Fear and Loathing in Las Vegas (1998)    5.0
7                              Pi (1998)    5.0
8                    Ginger Snaps (2000)    5.0
9   Ninja Scroll (Jûbei ninpûchô) (1995)    5.0

WEIGHTED:
                                        title  score
0                              Casino (1995)    5.0
1         Scott Pilgrim vs. the World (2010)    5.0
2                          Zombieland (2009)    5.0
3                Inglourious Basterds (2009)    5.0
4  Twelve Monkeys (a.k.a. 12 Monkeys) (1995)    5.0
5             Gods Must Be Crazy, The (1980)    5.0
6                     Rosemary's Baby (1968)    5.0
7                          Swin

In [39]:
user_item_filled = user_item_matrix.fillna(0)

In [78]:
from sklearn.decomposition import TruncatedSVD

# fill missing values with 0
user_item_filled = user_item_matrix.fillna(0)

# apply SVD
svd = TruncatedSVD(n_components=50, random_state=42)
latent_matrix = svd.fit_transform(user_item_filled)

print("Latent matrix shape:", latent_matrix.shape)

Latent matrix shape: (610, 50)


In [79]:
reconstructed = latent_matrix.dot(svd.components_)

reconstructed_df = pd.DataFrame(
    reconstructed,
    index=user_item_matrix.index,
    columns=user_item_matrix.columns
)

In [80]:
def recommend_svd(user_id, n=10):
    user_ratings = reconstructed_df.loc[user_id]
    
    # movies already watched
    seen_movies = user_item_matrix.loc[user_id].dropna().index
    
    # remove watched movies
    recommendations = user_ratings[~user_ratings.index.isin(seen_movies)]
    
    # top N
    return recommendations.sort_values(ascending=False).head(n)

In [81]:
svd_recs = recommend_svd(1)

svd_df = svd_recs.reset_index()
svd_df.columns = ['movieId', 'score']
svd_df = pd.merge(svd_df, movies, on='movieId')

print(svd_df[['title', 'score']])

                                       title     score
0                            Die Hard (1988)  3.541428
1                      Godfather, The (1972)  3.255015
2                 Breakfast Club, The (1985)  2.934956
3                             Aladdin (1992)  2.769484
4  Twelve Monkeys (a.k.a. 12 Monkeys) (1995)  2.683407
5                              Snatch (2000)  2.667320
6             Godfather: Part II, The (1974)  2.621964
7          Terminator 2: Judgment Day (1991)  2.584115
8                          Armageddon (1998)  2.241011
9                  Lady and the Tramp (1955)  2.136995


In [82]:
test_user_items = test.groupby('userId')['movieId'].apply(set).to_dict()

In [98]:
def precision_at_k(model_func, k=10):
    precisions = []
    
    for user in test_user_items.keys():
        if user not in user_item_matrix.index:
            continue
        
        # recommended movies
        recs = model_func(user, n=k)
        rec_ids = set(recs.index)
        
        # actual relevant movies
        true_items = set(
            test[(test['userId'] == user) & (test['rating'] >= 3)]['movieId']
        )
        
        if len(rec_ids) == 0:
            continue
        
        precision = len(rec_ids & true_items) / len(rec_ids)
        precisions.append(precision)
    
    return sum(precisions) / len(precisions)

In [99]:
def baseline_model(user_id, n=10):
    return top_10_filtered['mean_rating']

def cf_model(user_id, n=10):
    return recommend_movies_weighted(user_id, n)

def svd_model(user_id, n=10):
    return recommend_svd(user_id, n)

In [100]:
print("Baseline Precision@5:", precision_at_k(baseline_model, 5))
print("CF Precision@5:", precision_at_k(cf_model, 5))
print("SVD Precision@5:", precision_at_k(svd_model, 5))

Baseline Precision@5: 0.050491803278688525
CF Precision@5: 0.02754098360655738
SVD Precision@5: 0.3111475409836066


In [102]:
print("Baseline Precision@10:", precision_at_k(baseline_model, 10))
print("CF Precision@10:", precision_at_k(cf_model, 10))
print("SVD Precision@10:", precision_at_k(svd_model, 10))

Baseline Precision@10: 0.050491803278688525
CF Precision@10: 0.02573770491803279
SVD Precision@10: 0.260327868852459
